In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
CSV_PATH = "raw_loan_dataset.csv"
print(CSV_PATH)

raw_loan_dataset.csv


In [3]:
# --------------------------------
# 1) Load + initial snapshot
# --------------------------------
df = pd.read_csv(CSV_PATH)

print("\n=== INITIAL HEAD ===")
print(df.head())

print("\n=== INITIAL INFO ===")
print(df.info())

print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())



=== INITIAL HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  

=== INITIAL INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount 

In [4]:
# --------------------------------
# 2) Clean currency formatting
# --------------------------------
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

In [15]:
# --------------------------------
# 3) Normalize categorical typos BEFORE imputation
# --------------------------------
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
print(df["HasCollateral"].value_counts(dropna=False))
print("=================")
print(df["PreviousDefaults"].value_counts(dropna=False))
print("=================")
print(df["Approved"].value_counts(dropna=False) )



HasCollateral
No     50
Yes    48
NaN     2
Name: count, dtype: int64
PreviousDefaults
No     83
Yes    15
NaN     2
Name: count, dtype: int64
Approved
Yes    66
No     34
Name: count, dtype: int64


In [27]:
# --------------------------------
# 4) Impute missing values
# --------------------------------
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])



In [7]:
# --------------------------------
# 5) Remove duplicates
# --------------------------------
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")



Dropped duplicates: 103 -> 100 rows


In [21]:
# --------------------------------
# 6) IQR capping on numeric columns
# L3: clip extreme values to the IQR fence — same idea as house data-processing.py
# --------------------------------
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)
print(df["Income"])

0     108810.0
1      96482.0
2      28478.0
3      25851.0
4      38396.0
        ...   
95     66359.0
96     61241.0
97     63829.0
98     97529.0
99     62268.0
Name: Income, Length: 100, dtype: float64


In [22]:
# --------------------------------
# 7) Label encoding (Yes/No → 0/1)
# L3: same mapping for target (Approved) and two-category features
# Approved=1, Rejected=0
# --------------------------------
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))



=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [43]:
# --------------------------------
# 8) Class balance check
# L3/L5: imbalanced classes can make Accuracy misleading
# --------------------------------
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().map({"Yes": 1, "No": 0}).fillna(0).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().map({"Yes": 1, "No": 0}).fillna(0).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


In [45]:
# --------------------------------
# 9) Feature engineering (no leakage from label)
# --------------------------------
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              0   
1   96482.0        524.0             15.0     11200.0              0   
2   28478.0        640.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              0   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.237111           51814.285714  
1                 0         1      0.116084            6030.125000  
2                 0         1      0.424889            4449.687500  
3                 0         1      0.270783            1389.838710  
4                 0         1      0.278675            3555.185185  


In [47]:
# --------------------------------
# 11) Final snapshot + save
# Features (X) = all columns except Approved; label (y) = Approved
# --------------------------------
print("\n=== FINAL HEAD ===")
print(df.head())

print("\n=== FINAL INFO ===")
print(df.info())

print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())
OUT_PATH="clean_loan_dataset.csv"
print(df.to_csv(OUT_PATH))


=== FINAL HEAD ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              0   
1   96482.0        524.0             15.0     11200.0              0   
2   28478.0        640.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              0   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.237111           51814.285714  
1                 0         1      0.116084            6030.125000  
2                 0         1      0.424889            4449.687500  
3                 0         1      0.270783            1389.838710  
4                 0         1      0.278675            3555.185185  

=== FINAL INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column    